# Decision Tree Regressor
- Splits data into smaller and smaller groups using YES/NO questions
- Each split picks the feature/threshold that reduces error the most
- Leaf node = final answer = mean of all points that fell there
- Problem: unlimited depth = memorizes training data (overfit)
- Fix: pre-pruning (limit depth) or post-pruning (ccp_alpha)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.tree import DecisionTreeRegressor, plot_tree, export_text
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import mean_squared_error, r2_score

## 1. Understand impurity: entropy (classification) vs variance (regression)

In [ ]:
def entropy(labels):
    _, counts = np.unique(labels, return_counts=True)
    p = counts / counts.sum()
    return -(p * np.log2(p)).sum()

print("Entropy of [0,0,1,1]  :", entropy([0, 0, 1, 1]))        # 1.0 -> maximum impurity
print("Entropy of [0,0,0,0]  :", entropy([0, 0, 0, 0]) + 0.0)  # 0.0 -> pure node
print("Variance (regression) :", np.var([10, 12, 11, 40]).round(2))

## Impurity Measures
- Classification → Entropy (or Gini): measures class mixing
- Regression → Variance (MSE): measures spread of values
- Both measure: how "messy" is this group?
- Goal of each split: make child nodes LESS messy than parent
- Entropy=0 or Variance=0 → perfect pure node

## 2. Manual first split: try every threshold and pick the one with lowest weighted MSE

In [ ]:
rng = np.random.RandomState(1)
X = np.sort(rng.uniform(0, 6, 120)).reshape(-1, 1)
y = np.sin(X).ravel() + rng.normal(0, 0.25, 120)

def best_split(x, y):
    best = (None, np.inf)
    for t in (x[:-1] + x[1:]) / 2:
        left, right = y[x <= t], y[x > t]
        score = (len(left) * left.var() + len(right) * right.var()) / len(y)
        if score < best[1]:
            best = (t, score)
    return best

t, s = best_split(X.ravel(), y)
print(f"Best first split: x <= {t:.3f}  (weighted MSE = {s:.4f}, before split = {y.var():.4f})")
print("sklearn root threshold:", DecisionTreeRegressor(max_depth=1).fit(X, y).tree_.threshold[0].round(3))

x[:-1] → all points except last

x[1:] → all points except first

Adding them and dividing by 2 → midpoints between consecutive points

## Manual First Split
- Try every midpoint between consecutive x values as threshold
- For each threshold: compute weighted variance of left + right groups
- Pick threshold with LOWEST weighted variance
- This is exactly what sklearn does internally
- Verify: our threshold matches sklearn's tree_.threshold[0]

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=0)
X_plot = np.linspace(0, 6, 400).reshape(-1, 1)

## 3. Overfitting: effect of max_depth

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(18, 4))
for ax, d in zip(axes, [1, 3, 5, None]):
    m = DecisionTreeRegressor(max_depth=d, random_state=0).fit(X_train, y_train)
    ax.scatter(X_train, y_train, s=10)
    ax.plot(X_plot, m.predict(X_plot), "r")
    ax.set_title(f"depth={d} | train R2={m.score(X_train, y_train):.2f}, test R2={m.score(X_test, y_test):.2f}")
plt.show()

## max_depth Effect
- depth=1  → 2 leaves → too simple (underfit)
- depth=3  → good balance (sweet spot)
- depth=5  → starts memorizing noise
- depth=None → train R2=1.0 but test R2 collapses (overfit)
- Tree predictions look like stairs (step function)
- More depth = more steps = closer fit = more overfit risk

## 4. Pruning
**Pre-pruning:** max_depth, min_samples_split, min_samples_leaf.
**Post-pruning (cost-complexity):** grow the full tree, then remove weak branches using ccp_alpha.

In [ ]:
full = DecisionTreeRegressor(random_state=0)
path = full.cost_complexity_pruning_path(X_train, y_train)
alphas = path.ccp_alphas[:-1]
trees = [DecisionTreeRegressor(random_state=0, ccp_alpha=a).fit(X_train, y_train) for a in alphas]

plt.plot(alphas, [t.score(X_train, y_train) for t in trees], label="train R2")
plt.plot(alphas, [t.score(X_test, y_test) for t in trees], label="test R2")
plt.xlabel("ccp_alpha"); plt.legend(); plt.title("Post-pruning: pick alpha where test R2 peaks")
plt.show()
best_alpha = alphas[int(np.argmax([t.score(X_test, y_test) for t in trees]))]
print("Best ccp_alpha:", round(best_alpha, 5))

## Post-Pruning (ccp_alpha)
- Grow full overfit tree first
- cost_complexity_pruning_path → finds all useful alpha values
- Higher alpha → more branches removed → simpler tree
- Plot train R2 vs test R2 across alphas
- Pick alpha where test R2 peaks
- Pre-pruning = stop early, Post-pruning = grow then cut

## 5. Tune pre-pruning params with GridSearchCV and evaluate

In [ ]:
grid = GridSearchCV(DecisionTreeRegressor(random_state=0),
                    {"max_depth": [2, 3, 4, 5, 6, None], "min_samples_leaf": [1, 3, 5, 10],
                     "ccp_alpha": [0, 0.001, 0.005, 0.01]},
                    cv=5, scoring="neg_mean_squared_error").fit(X_train, y_train)
best = grid.best_estimator_
pred = best.predict(X_test)
print("Best params:", grid.best_params_)
print(f"Test MSE={mean_squared_error(y_test, pred):.4f} | R2={r2_score(y_test, pred):.4f}")

## 6. Visualize the tree

In [ ]:
small = DecisionTreeRegressor(max_depth=3, random_state=0).fit(X_train, y_train)
plt.figure(figsize=(14, 6))
plot_tree(small, feature_names=["x"], filled=True, rounded=True)
plt.show()
print(export_text(small, feature_names=["x"]))

## Visualizing the Tree
- plot_tree → visual boxes connected by branches
- export_text → same tree as if/else rules
- Each node shows: split rule, MSE, sample count, prediction
- filled=True → darker color = more extreme prediction value
- max_depth=3 for visualization (deeper = unreadable)
- Prediction = follow rules from root to leaf → return leaf value

## Decision Tree Rules to Remember
1. Splits by minimizing weighted variance (regression)
2. More depth = follows noise = overfit
3. Prediction = mean of all training points in that leaf
4. Pre-pruning: max_depth, min_samples_leaf (stop early)
5. Post-pruning: ccp_alpha (grow full tree then prune)
6. Pick ccp_alpha where test R2 peaks on pruning plot
7. Tree predictions look like stairs (step function)
8. No scaling needed (trees are not distance-based)
9. export_text shows the exact if/else rules learned
10. Support vectors → trees use ALL points, not just boundary ones